In [1]:
pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests

url = "http://books.toscrape.com/"
response = requests.get(url)

print(response.status_code)  # should be 200

200


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

products = soup.find_all("article", class_="product_pod")
print(len(products))

20


In [4]:
data = []

for product in products:
    title = product.h3.a["title"]
    price = product.find("p", class_="price_color").text
    availability = product.find("p", class_="instock availability").text.strip()
    
    rating_class = product.p["class"][1]  # e.g. 'Three'
    
    data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating_class
    })

In [5]:
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

all_data = []

for page in range(1, 6):  # scrape first 5 pages
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    products = soup.find_all("article", class_="product_pod")
    
    for product in products:
        title = product.h3.a["title"]
        price = product.find("p", class_="price_color").text
        
        all_data.append({
            "title": title,
            "price": price
        })

In [6]:
import pandas as pd

df = pd.DataFrame(all_data)
df.to_csv("books_data.csv", index=False)

print(df.head())

                                   title    price
0                   A Light in the Attic  Â£51.77
1                     Tipping the Velvet  Â£53.74
2                             Soumission  Â£50.10
3                          Sharp Objects  Â£47.82
4  Sapiens: A Brief History of Humankind  Â£54.23


In [8]:
df["price"] = (
    df["price"]
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [9]:
from datetime import datetime
df["scraped_at"] = datetime.now()

In [10]:
try:
    response = requests.get(url)
    response.raise_for_status()
except Exception as e:
    print("Error:", e)